In [1]:
import slangpy as spy
from pyglm import glm
import matplotlib.pyplot as plt
import numpy as np

from bvhgs import device
from bvhgs.camera import Camera
from bvhgs.gaussian import GaussianCloud
from bvhgs.renderer import Renderer
from bvhgs.prefix_sum import prefix_sum
import pathlib

[INFO] (rhi) layer: CreateDevice: Debug layer is enabled.
[WARN] No supported shader model found, pretending to support sm_6_0.


In [2]:
np.random.seed(348)

# Create Gaussian Buffer

In [3]:
gaussians = GaussianCloud()
gaussians.load_from_ply(pathlib.Path("../resources/scenes/truck.ply"))
len(gaussians)

148916

# Load Module and Shader

In [4]:
module = device.load_module("renderer.slang")
module

SlangModule(
  name = renderer.slang,
  path = /Users/fangjun/Documents/stanford/bvhgs/src/bvhgs/slang/renderer.slang,
  entry_points = [
    SlangEntryPoint(name="project", stage=compute),
    SlangEntryPoint(name="cull", stage=compute),
    SlangEntryPoint(name="computeTile", stage=compute),
    SlangEntryPoint(name="buildGaussianTable", stage=compute),
    SlangEntryPoint(name="duplicateGaussian", stage=compute),
    SlangEntryPoint(name="rasterize", stage=compute),
  ]
)

In [5]:
program = device.link_program([module], [])
program

ShaderProgram(
  modules = [
    SlangModule(
      name = renderer.slang,
      path = /Users/fangjun/Documents/stanford/bvhgs/src/bvhgs/slang/renderer.slang,
      entry_points = [
        SlangEntryPoint(name="project", stage=compute),
        SlangEntryPoint(name="cull", stage=compute),
        SlangEntryPoint(name="computeTile", stage=compute),
        SlangEntryPoint(name="buildGaussianTable", stage=compute),
        SlangEntryPoint(name="duplicateGaussian", stage=compute),
        SlangEntryPoint(name="rasterize", stage=compute),
      ]
    ),
  ],
  entry_points = []
)

# Projection

## Build Slang Buffer and Camera Parameter

In [6]:
# Create a buffer for the Gaussian points.
gaussian_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_gaussian_3d,
    usage=spy.BufferUsage.shader_resource,
)
# Store all the gaussian points in the buffer.
gaussian_cursor = spy.BufferCursor(
    program.reflection.g_gaussian_3d.type_layout.element_type_layout,
    gaussian_buf,
)

for i in range(len(gaussians)):
    gaussian_cursor[i].write(gaussians[i])
gaussian_cursor.apply()

In [7]:
# Create a buffer for the Gaussian2D points.
gaussian2d_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_gaussian_2d,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)

In [8]:
camera = Camera(
    rotation=glm.quat(1, 0, 0, 0),
    translation=glm.vec3(0, 0, 1),
    sensor_size=glm.uvec2(512, 512),
    focal_length=128
)
camera.to_slang()

{'_rotation': [0.0, 0.0, 0.0, 1.0],
 '_translation': vec3( 0, 0, 1 ),
 '_sensorSize': uvec2( 512, 512 ),
 '_focalLength': 128,
 '_nearPlane': 0.1,
 '_farPlane': 1000.0}

## Dispatch Projection Kernel

In [9]:
ker_proj = device.create_compute_kernel(
    device.link_program([module], [module.entry_point("project")])
)
ker_proj

ComputeKernel(0x600000eb61c0)

In [10]:
# Create flag buffer for culling.
inside_flag_buf = device.create_buffer(
    element_count=len(gaussians),
    struct_type=program.reflection.g_inside_flag,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)

In [11]:
ker_proj.dispatch(
    thread_count=[len(gaussians), 1, 1],
    vars={
        "g_camera": camera.to_slang(),
        "g_gaussian_3d": gaussian_buf,
        "g_gaussian_2d": gaussian2d_buf,
        "g_inside_flag": inside_flag_buf
    }
)

In [12]:
%%timeit
ker_proj.dispatch(
    thread_count=[len(gaussians), 1, 1],
    vars={
        "g_camera": camera.to_slang(),
        "g_gaussian_3d": gaussian_buf,
        "g_gaussian_2d": gaussian2d_buf,
        "g_inside_flag": inside_flag_buf
    }
)

628 μs ± 1.61 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [13]:
cursor = spy.BufferCursor(
    program.reflection.g_gaussian_2d.type_layout.element_type_layout,
    gaussian2d_buf,
)
cursor.element_count

148916

## Cull Gaussians

In [14]:
ker_cull = device.create_compute_kernel(
    device.link_program([module], [module.entry_point("cull")])
)

In [15]:
# Cull the Gaussian points.
inside_offset_buf = prefix_sum(inside_flag_buf)
inside_offset_cursor = spy.BufferCursor(
    program.reflection.g_inside_offset.type_layout.element_type_layout,
    inside_offset_buf,
)
# Read the last element of the cull prefix buffer to get the number of culled points.
num_viewing = int(inside_offset_cursor[len(inside_offset_cursor) - 1].read())  # type: ignore
num_viewing

120350

In [16]:
%timeit prefix_sum(inside_flag_buf)

211 μs ± 25.4 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [17]:
# Culled gaussians
culled_gaussian_buf = device.create_buffer(
    element_count=num_viewing,
    struct_type=program.reflection.g_gaussian_2d_culled,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)

In [18]:
ker_cull.dispatch(
    thread_count=[len(gaussians), 1, 1],
    vars={
        "g_gaussian_2d": gaussian2d_buf,
        "g_inside_flag": inside_flag_buf,
        "g_inside_offset": inside_offset_buf,
        "g_gaussian_2d_culled": culled_gaussian_buf
    }
)

In [19]:
%%timeit
ker_cull.dispatch(
    thread_count=[len(gaussians), 1, 1],
    vars={
        "g_gaussian_2d": gaussian2d_buf,
        "g_inside_flag": inside_flag_buf,
        "g_inside_offset": inside_offset_buf,
        "g_gaussian_2d_culled": culled_gaussian_buf
    }
)

202 μs ± 1.75 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [20]:
cursor = spy.BufferCursor(
    program.reflection.g_gaussian_2d_culled.type_layout.element_type_layout,
    culled_gaussian_buf,
)
cursor[0]

{'position': {0.4848171, 0.5113666, 0.0011133038}, 'covariance': {{0.0003242842, -0.011305249}, {0.0054939636, 0.0050626216}}, 'color': {0.3868044, 0.39372838, 0.49184987}, 'opacity': 0.5294011235237122, 'cachedInvCov': {{79.410736, 177.33069}, {-86.176636, 5.086623}}} [Gaussian2D]

# Tiling

In [21]:
ker_tile = device.create_compute_kernel(
    device.link_program([module], [module.entry_point("computeTile")])
)
ker_build_gs = device.create_compute_kernel(
    device.link_program([module], [module.entry_point("buildGaussianTable")])
)

In [22]:
num_tile_buf = device.create_buffer(
    element_count=num_viewing,
    struct_type=program.reflection.g_num_tile,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)
gaussian_table_buf = device.create_buffer(
    element_count=num_viewing,
    struct_type=program.reflection.g_gaussian_table,
    usage=spy.BufferUsage.shader_resource | spy.BufferUsage.unordered_access,
)

In [23]:
ker_tile.dispatch(
    thread_count=[num_viewing, 1, 1],
    vars={
        "g_gaussian_2d_culled": culled_gaussian_buf,
        "g_num_tile": num_tile_buf
    }
)

In [24]:
%%timeit
ker_tile.dispatch(
    thread_count=[num_viewing, 1, 1],
    vars={
        "g_gaussian_2d_culled": culled_gaussian_buf,
        "g_num_tile": num_tile_buf
    }
)

64.9 μs ± 416 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [25]:
num_tile_prefix_buf = prefix_sum(num_tile_buf)

In [26]:
ker_build_gs.dispatch(
    thread_count=[num_viewing, 1, 1],
    vars={
        "g_gaussian_2d_culled": culled_gaussian_buf,
        "g_num_tile_prefix": num_tile_prefix_buf,
        "g_gaussian_table": gaussian_table_buf,
    }
)

In [27]:
%%timeit
ker_build_gs.dispatch(
    thread_count=[num_viewing, 1, 1],
    vars={
        "g_gaussian_2d_culled": culled_gaussian_buf,
        "g_num_tile_prefix": num_tile_prefix_buf,
        "g_gaussian_table": gaussian_table_buf,
    }
)

The slowest run took 6.91 times longer than the fastest. This could mean that an intermediate result is being cached.
28.5 μs ± 31.1 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
